# 🌳 Decision Tree Model Training

This notebook trains the Decision Tree baseline model.

**Prerequisites**: Run `01_preprocessing.ipynb` first to generate preprocessed data.

In [6]:
import sys, os
import warnings
warnings.filterwarnings('ignore')
import pickle

# Ensure PROJECT_P is on the path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd

# Project modules
from src.config import *
from src.utils import get_logger

logger = get_logger('Decision Tree')
print('✅ Imports successful')

✅ Imports successful


## 1. Load Preprocessed Data

In [7]:
# Load preprocessed data from file
preprocess_file = os.path.join(MODEL_DIR, 'preprocessed_data.pkl')

if not os.path.exists(preprocess_file):
    raise FileNotFoundError(f'Please run 01_preprocessing.ipynb first to generate {preprocess_file}')

with open(preprocess_file, 'rb') as f:
    prep = pickle.load(f)

X_train = prep['X_train']
X_val = prep['X_val']
X_test = prep['X_test']
y_train = prep['y_train']
y_val = prep['y_val']
y_test = prep['y_test']

print(f'✅ Preprocessed data loaded')
print(f'   Train: {X_train.shape}')
print(f'   Val:   {X_val.shape}')
print(f'   Test:  {X_test.shape}')

✅ Preprocessed data loaded
   Train: (124012, 94)
   Val:   (26575, 94)
   Test:  (26575, 94)


## 2. Train Decision Tree

In [8]:
from src.training import train_decision_tree

dt_model = train_decision_tree(X_train, y_train, X_val, y_val, use_smote=True)
print('\n✅ Decision Tree training complete')

18:37:23 | Training             | INFO    | ==================================================
18:37:23 | Training             | INFO    |   🌳 Training Decision Tree
18:37:23 | Training             | INFO    | ==================================================
18:37:23 | Training             | INFO    |   SMOTE Resampling:
18:37:23 | Training             | INFO    |     Before: 119,573 legit, 4,439 fraud (1:26.9)
18:37:23 | Training             | INFO    |     After:  119,573 legit, 59,786 fraud (1:2.0)
18:37:23 | Training             | INFO    |     Synthetic samples: 55,347
18:37:23 | Training             | INFO    | ⏳ Starting: Decision Tree training
18:37:23 | Models               | INFO    |   🌳 Decision Tree created (max_depth=12, class_weight=balanced)
18:37:30 | Training             | INFO    | ✅ Finished: Decision Tree training (6.6s)
18:37:30 | Training             | INFO    |   Val Accuracy: 0.9080
18:37:30 | Training             | INFO    |   Val F1-Score: 0.3036
18:37:30 |


✅ Decision Tree training complete


## 3. Evaluate on Test Set

## 3.5 Feature Importance Visualization

In [9]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

# Feature importance
feature_importance = dt_model.feature_importances_
feature_names = prep['feature_names'] if 'feature_names' in prep else [f'Feature {i}' for i in range(len(feature_importance))]

# Sort by importance
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(importance_df['feature'], importance_df['importance'], color='#2ecc71', edgecolor='black')
ax.set_xlabel('Importance Score', fontweight='bold')
ax.set_title('Decision Tree - Top 20 Most Important Features', fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig(os.path.join(EVAL_DIR, '02_dt_feature_importance.png'), dpi=150, bbox_inches='tight')
plt.show()
print('✅ Feature importance plot saved')

✅ Feature importance plot saved


In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# Predictions
y_pred = dt_model.predict(X_test)
y_pred_proba = dt_model.predict_proba(X_test)[:, 1]

# Metrics
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred, zero_division=0),
    'Recall': recall_score(y_test, y_pred, zero_division=0),
    'F1-Score': f1_score(y_test, y_pred, zero_division=0),
    'ROC-AUC': roc_auc_score(y_test, y_pred_proba),
}

print('\n' + '='*50)
print('Decision Tree - Test Set Performance')
print('='*50)
for metric, value in metrics.items():
    print(f'{metric:15s}: {value:.4f}')
print('='*50)


Decision Tree - Test Set Performance
Accuracy       : 0.9086
Precision      : 0.2085
Recall         : 0.5563
F1-Score       : 0.3033
ROC-AUC        : 0.7950


## 4. Save Model and Results

In [ ]:
# Save results (predictions only, not full model)
dt_results = {
    'y_pred': y_pred,
    'y_pred_proba': y_pred_proba,
    'metrics': metrics,
}

results_file = os.path.join(MODEL_DIR, 'decision_tree_results.pkl')
with open(results_file, 'wb') as f:
    pickle.dump(dt_results, f)

print(f'\n✅ Results saved to {results_file}')
print('\n📝 Next: Run 03_train_xgboost.ipynb or 04_train_hgnn.ipynb')



✅ Results saved to /Users/aaronr/Desktop/PROJECT_P/models/decision_tree_results.pkl

📝 Next: Run 03_train_xgboost.ipynb or 04_train_hgnn.ipynb
